# **The Win Condition: Exploring the Impact of Magic: The Gathering Card Features on Deck Win Rates**

**Abby Ortego**

**Milestone 1**

**[Github Webpage](https://abbyortego.github.io/cmps-6790-final-tutorial/)**

In [34]:
import requests
from bs4 import BeautifulSoup
import regex as re
import pandas as pd

## Project Goal & Plan

[Magic: The Gathering (MTG)](https://magic.wizards.com/en/intro) is a popular, fantasy-based trading card game with competitive tournaments hosted around the globe. By amassing a vast collection of impressive cards, a player can construct their own custom decks from their collection. During deck construction, players must adhere to a deck size limit while maintaining a balance between the types of interaction and resource cards. This trade-off makes for an interesting analysis into what is most important when constructing a game winning deck. 

This notebook aims to answer the following questions: 
1. Do certain features of a deck correlate with the game outcome? 
2. If so, which features have the biggest impact in determining a winning outcome for a player?  
3. Using those features, can we predict the probability of winning a game from just a deck? 

In Milestone 1, I plan to start tackling these questions by first exploring public Magic: The Gathering tournament data that lists the ranks of players and their decks. In Milestone 2, I plan to enrich the deck dataset curated in Milestone 1 by retrieving comprehensive card data from a reliable API source (e.g., [MTGJSON](https://mtgjson.com/), [Scryfall](https://scryfall.com/docs/api)). In the final notebook, I plan to leverage the enriched dataset to train a model that will accept features of a deck, constructed from the aggregate card data, and output the probability of a deck winning. 

## Dataset

The "winability" of a deck is best observed in tournaments which are held around the world, drawing the attention of a variety of players. When a tournament has ended, the host reports participating players, the decks used, tournament outcomes, and more. 

One site that collects and displays this information is [MTGTop8](https://mtgtop8.com) which sources its data from reliable reporting services (e.g., [Melee](https://melee.gg/), [Magic Events](https://magic.gg/), etc.). However, the site has no public facing API, so it must be scraped. Thankfully, it's [Privacy Policy](https://mtgtop8.com/privacy) and [robots.txt](https://mtgtop8.com/robots.txt) file does not ban web scrapers. 

MTGTop8 offers a lot of data on it's site and not all of it is relevant to predicting deck winability. Due to this, we'll focus on only a subset of tournaments and their features. 
- Standard Tournament Format
    - MTG was released in August 1993 and has tournaments hosted around the globe. This has lead to the evolution of a variety of play formats. We will be exploring decks played in standard formatted tournaments. 
    - The standard format is very popular and constrains deck construction to be only from cards released within the last 2-3 years. 
    - By selecting this format, we can avoid the possibility of more recent decks with new fancy abilities overpowering older decks and some aspects of the experience gap that can exist between players.
- Major Events
    - Even though web scraping is not banned by this site, it's still important to be mindful of the number of queries issued. Due to this, we will only be querying data from tournaments listed as "major events". 
    - They're not as common but have a high turn out which will still produce a reliable and large deck dataset with fewer queries. 
- Within the Last Full Month (January 2026 at the time of submission)
    - Rulings and strategies for MTG evolve over time so selecting tournaments that occurred recently will best reflect the current what aspects of a deck are most important for winning.   

In summary, this notebook will be extracting and analyzing Magic: The Gathering tournaments from MTGTop8 that are major events played in the standard format within the last full month (January 2026). 

# Extracting, Transforming, and Loading the Data

MTGTop8 organizes it's deck data by tournament, so we'll have to scrape tournament data first then visit each tournament page to collect deck related information. 

## Scraping Tournament Data

### Extracting Tournaments

On this [page](https://mtgtop8.com/format?f=ST&meta=46&a=) under the "Last 20 Events" header, major events played in the standard format within the last two months are listed. Unfortunately, these tournaments are paginated and only listed 20 at a time, so we'll have to query for each of these pages using the `cp=` parameter in the URL. 

In [35]:
# get all tournaments tables
tournaments_tables = []
for page in range(1, 8):    # there are roughly 7 pages worth of data
    r = requests.get(f"https://mtgtop8.com/format?f=ST&meta=46&cp={page}")
    if r.status_code != 200:    # checks status code just in case we run into issues
        print(f"Could not scrape events from page {page}.")
        continue
    #

    soup = BeautifulSoup(r.content)
    tables = soup.find_all("table")     # using soup to grab all tables on the page
    tournaments_tables.append(tables[2])   # the table we're interested in is the third one on the page 
#

for row in tournaments_tables[1].find_all('tr')[:2]:    # displays the html for the first 2 rows of the first table
    print(row.prettify())
#

<tr class="hover_tr">
 <td align="center" width="5%">
  <img height="17" src="/graph/online/paper.png" title="Paper"/>
 </td>
 <td class="S14" width="70%">
  <a href="event?e=80805&amp;f=ST">
   Friday LCQ #11
  </a>
  @
  <a class="und" href="event?e=80805&amp;f=ST">
   SCG CON Milwaukee
  </a>
 </td>
 <td align="center" width="13%">
  <img src="/graph/star.png"/>
  <img src="/graph/star.png"/>
 </td>
 <td align="right" class="S12" width="12%">
  20/02/26
 </td>
</tr>

<tr class="hover_tr">
 <td align="center" width="5%">
  <img height="17" src="/graph/online/paper.png" title="Paper"/>
 </td>
 <td class="S14" width="70%">
  <a href="event?e=80793&amp;f=ST">
   Friday LCQ #10
  </a>
  @
  <a class="und" href="event?e=80793&amp;f=ST">
   SCG CON Milwaukee
  </a>
 </td>
 <td align="center" width="13%">
  <img src="/graph/star.png"/>
  <img src="/graph/star.png"/>
 </td>
 <td align="right" class="S12" width="12%">
  20/02/26
 </td>
</tr>



Now that we have the tournament tables, we can extract the information we're interested in from the html. We're looking for...
1. Tournament Names (to identify tournaments) and Dates (to grab recent tournaments)
    - These are stored as text within the html tags, so we can use BeautifulSoup's `get_text()` method to extract this information from the table. 
    - When you do you get a string that looks like this: `'\n\n\nRCQ @ Draco Hobby Center (Bogota, Colombia)\n\n01/02/26\n\n\n\nMTGO RC Super Qualifier\n\n01/02/26\n\n\n\n2nd Chance PTQ @ Pro Tour Lorwyn Eclipsed (Richmond, VA)\n\n31/01/26\n\n\n\n`
    - It's not very pretty but the new line characters do have a pattern! 3-4 new line characters separate tournaments and 2 new line characters separate tournament name from date. We can use regex to parse the string using this pattern. 
2. Links to Additional Tournament Information (grab more details later)
    - These are stored in `a` tags with an `href` property
    - Some entries in the table record an additional `a` tag with the `class` property specified in addition to `href`. These tags have duplicate information that we don't need. 
    - We can use BeautifulSoup's `find_all()` method to find all `a` tags that have an `href` property specified but no `class` property.- Lastly, we can use the `get()` method to extract the links. 

In [ ]:
# lists to store data
names = []
dates = []
links = []

for tournaments_table in tournaments_tables:
    # (1) Tournament Names and Date
    table_text = tournaments_table.get_text()
    matches = re.finditer(r"""
        (?:\n){3,4}(?P<name>.+)         # 3-4 '\n' (non-captured) followed by a tournament name
        (?:\n){2}(?P<date>.+)       # 2 '\n' (non-captured) followed by a date
        """, 
        table_text, 
        re.VERBOSE      # for multiline to support comments!
    )  
    for match in matches:
        names.append(match.group('name'))
        dates.append(match.group('date'))
    #

    # (2) Links to Additional Tournament Information
    table_hrefs = tournaments_table.find_all("a", href=True, class_=False)      # find all 'a' tags with an 'href' but no 'class'
    links.extend([table_href.get("href") for table_href in table_hrefs])        # get the link
#

display(names[:5], dates[:5], links[:5])

['MTGO Challenge 32 NEW',
 'MTGO Challenge 64 NEW',
 'Champions Cup Store Qualifier @ Machida (Japan) NEW',
 'MTGO Challenge 64',
 'Sunday ReCQ #2 @ SCG CON Milwaukee NEW']

['26/02/26', '24/02/26', '23/02/26', '23/02/26', '22/02/26']

['event?e=81053&f=ST',
 'event?e=81055&f=ST',
 'event?e=81001&f=ST',
 'event?e=80924&f=ST',
 'event?e=80950&f=ST']

Finally, we can store this information into a pandas dataframe!

In [161]:
all_tournaments_df = pd.DataFrame({'Name': names, 'Date': dates, 'Link': links})
display(all_tournaments_df.head(), all_tournaments_df.dtypes)

,Name,Date,Link
0,MTGO Challenge 32 NEW,26/02/26,event?e=81053&f=ST
1,MTGO Challenge 64 NEW,24/02/26,event?e=81055&f=ST
2,Champions Cup Store Qualifier @ Machida (Japan...,23/02/26,event?e=81001&f=ST
3,MTGO Challenge 64,23/02/26,event?e=80924&f=ST
4,Sunday ReCQ #2 @ SCG CON Milwaukee NEW,22/02/26,event?e=80950&f=ST


Name    str
Date    str
Link    str
dtype: object

### Transforming Tournaments Table

The dataframe we got in the [Extracting](#extracting-tournaments) section above is very close to what we're interested in retrieving as detailed in the [Dataset](#dataset) section, but a few adjustments are still required.

1. No online tournaments

Some of these tournaments include MTG games played online in the app which has a similar but alternative system for collecting cards and creating decks. Due to this we want to only look at tournaments played in person. Thankfully, these online tournaments are denoted with "MTGO" so we can filter the `Name` property of the `tournaments_df` to only include tournaments without "MTGO" in the name. 

In [162]:
# boolean mask of tournament names that do not include MTGO
not_online = ~all_tournaments_df["Name"].str.contains("MTGO")       
all_tournaments_df = all_tournaments_df[not_online].reset_index(drop=True)
display(all_tournaments_df.head())      # only in person tournaments!

,Name,Date,Link
0,Champions Cup Store Qualifier @ Machida (Japan...,23/02/26,event?e=81001&f=ST
1,Sunday ReCQ #2 @ SCG CON Milwaukee NEW,22/02/26,event?e=80950&f=ST
2,Sunday RCQ #1 @ SCG CON Milwaukee NEW,22/02/26,event?e=80949&f=ST
3,Saturday RCQ #2 @ SCG CON Milwaukee,21/02/26,event?e=80908&f=ST
4,Saturday RCQ #1 @ SCG CON Milwaukee,21/02/26,event?e=80907&f=ST


2. Recent Tournaments

This dataframe contains tournaments from the last **2** months, we are only concerned with tournaments that occurred in the last full month (January 2026). We'll filter out tournaments from before or after that time and convert our `Date` column in the pandas dataframe to the proper type (datetime) to do this.

In [163]:
all_tournaments_df["Date"] = pd.to_datetime(all_tournaments_df["Date"], format="%d/%m/%y")      # format to datetime
# boolean mask of rows between the start & end of January
in_january = (all_tournaments_df["Date"] >= '2026-01-01') & (all_tournaments_df["Date"] < '2026-02-01')         
all_tournaments_df = all_tournaments_df[in_january].reset_index(drop=True)
display(all_tournaments_df.head())      # only in January!

,Name,Date,Link
0,2nd Chance PTQ @ Pro Tour Lorwyn Eclipsed (Ric...,2026-01-31,event?e=79854&f=ST
1,Champions Cup Special Qualifier @ Kawasaki (Ja...,2026-01-30,event?e=79785&f=ST
2,Pro Tour Lorwyn Eclipsed @ Richmond,2026-01-30,event?e=79746&f=ST
3,Saturday ReCQ #2 @ SCG CON Portland,2026-01-25,event?e=79664&f=ST
4,Super Sunday RCQ #2 @ SCG CON Portland,2026-01-25,event?e=79600&f=ST


3. Base URL for the Link

The `Link` we extracted contains only the suffix of the URL. We need to append the base URL ("https://mtgtop8.com") to this column for easy access later.

In [164]:
base_url = "https://mtgtop8.com"
# concatenate the base URL string with a '/' then the suffix
all_tournaments_df["Link"] = base_url + '/' + all_tournaments_df["Link"]
display(all_tournaments_df.head())      # proper links!

,Name,Date,Link
0,2nd Chance PTQ @ Pro Tour Lorwyn Eclipsed (Ric...,2026-01-31,https://mtgtop8.com/event?e=79854&f=ST
1,Champions Cup Special Qualifier @ Kawasaki (Ja...,2026-01-30,https://mtgtop8.com/event?e=79785&f=ST
2,Pro Tour Lorwyn Eclipsed @ Richmond,2026-01-30,https://mtgtop8.com/event?e=79746&f=ST
3,Saturday ReCQ #2 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79664&f=ST
4,Super Sunday RCQ #2 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79600&f=ST


4. Overloaded Name Column

The tournament name is storing a lot of information (e.g., day, actual name, and convention). We can use regex to separate the values into their own columns.

In [165]:
all_tournaments_df[:10]

,Name,Date,Link
0,2nd Chance PTQ @ Pro Tour Lorwyn Eclipsed (Ric...,2026-01-31,https://mtgtop8.com/event?e=79854&f=ST
1,Champions Cup Special Qualifier @ Kawasaki (Ja...,2026-01-30,https://mtgtop8.com/event?e=79785&f=ST
2,Pro Tour Lorwyn Eclipsed @ Richmond,2026-01-30,https://mtgtop8.com/event?e=79746&f=ST
3,Saturday ReCQ #2 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79664&f=ST
4,Super Sunday RCQ #2 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79600&f=ST
5,Super Sunday RCQ #1 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79599&f=ST
6,Izzet Explosive Experiment Event,2026-01-25,https://mtgtop8.com/event?e=79598&f=ST
7,Regional Championship @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79462&f=ST
8,Saturday RCQ @ SCG CON Portland,2026-01-24,https://mtgtop8.com/event?e=79569&f=ST
9,"RCQ @ BB-Spiele (Rosenheim, Germany)",2026-01-24,https://mtgtop8.com/event?e=79428&f=ST


In [166]:
unloaded_name_col = all_tournaments_df['Name'].str.extract(r"""
    (?:Monday\s|Tuesday\s|Wednesday\s|Thursday\s|Friday\s|Saturday\s|Sunday\s)?         # non-capture the day
    (?P<Name>[^@]+)         # capture everything up until the @ symbol
    (?:@)?      # non-capture the @ symbol, if it exists (indicated with ?)
    (?P<Convention>.+)?         # capture convention, if it exists (indicated with ?)
    """,
    re.VERBOSE
)
display(unloaded_name_col[:10])

,Name,Convention
0,2nd Chance PTQ,"Pro Tour Lorwyn Eclipsed (Richmond, VA)"
1,Champions Cup Special Qualifier,Kawasaki (Japan)
2,Pro Tour Lorwyn Eclipsed,Richmond
3,ReCQ #2,SCG CON Portland
4,Super Sunday RCQ #2,SCG CON Portland
5,Super Sunday RCQ #1,SCG CON Portland
6,Izzet Explosive Experiment Event,NaN
7,Regional Championship,SCG CON Portland
8,RCQ,SCG CON Portland
9,RCQ,"BB-Spiele (Rosenheim, Germany)"


In [167]:
# inner join the two dataframes on their index columns
all_tournaments_df = all_tournaments_df.merge(unloaded_name_col, right_index=True, left_index=True)
display(all_tournaments_df[:10])

,Name_x,Date,Link,Name_y,Convention
0,2nd Chance PTQ @ Pro Tour Lorwyn Eclipsed (Ric...,2026-01-31,https://mtgtop8.com/event?e=79854&f=ST,2nd Chance PTQ,"Pro Tour Lorwyn Eclipsed (Richmond, VA)"
1,Champions Cup Special Qualifier @ Kawasaki (Ja...,2026-01-30,https://mtgtop8.com/event?e=79785&f=ST,Champions Cup Special Qualifier,Kawasaki (Japan)
2,Pro Tour Lorwyn Eclipsed @ Richmond,2026-01-30,https://mtgtop8.com/event?e=79746&f=ST,Pro Tour Lorwyn Eclipsed,Richmond
3,Saturday ReCQ #2 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79664&f=ST,ReCQ #2,SCG CON Portland
4,Super Sunday RCQ #2 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79600&f=ST,Super Sunday RCQ #2,SCG CON Portland
5,Super Sunday RCQ #1 @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79599&f=ST,Super Sunday RCQ #1,SCG CON Portland
6,Izzet Explosive Experiment Event,2026-01-25,https://mtgtop8.com/event?e=79598&f=ST,Izzet Explosive Experiment Event,NaN
7,Regional Championship @ SCG CON Portland,2026-01-25,https://mtgtop8.com/event?e=79462&f=ST,Regional Championship,SCG CON Portland
8,Saturday RCQ @ SCG CON Portland,2026-01-24,https://mtgtop8.com/event?e=79569&f=ST,RCQ,SCG CON Portland
9,"RCQ @ BB-Spiele (Rosenheim, Germany)",2026-01-24,https://mtgtop8.com/event?e=79428&f=ST,RCQ,"BB-Spiele (Rosenheim, Germany)"


Since both dataframes had a `Name` column, the merge produces `Name_x`, our old overloaded column, and `Name_y`, our new unloaded column. We no longer need `Name_x` so we can drop it.

In [168]:
all_tournaments_df.drop('Name_x', axis=1, inplace=True)
all_tournaments_df.rename(columns={"Name_y": "Name"}, inplace=True)
display(all_tournaments_df[:10])

,Date,Link,Name,Convention
0,2026-01-31,https://mtgtop8.com/event?e=79854&f=ST,2nd Chance PTQ,"Pro Tour Lorwyn Eclipsed (Richmond, VA)"
1,2026-01-30,https://mtgtop8.com/event?e=79785&f=ST,Champions Cup Special Qualifier,Kawasaki (Japan)
2,2026-01-30,https://mtgtop8.com/event?e=79746&f=ST,Pro Tour Lorwyn Eclipsed,Richmond
3,2026-01-25,https://mtgtop8.com/event?e=79664&f=ST,ReCQ #2,SCG CON Portland
4,2026-01-25,https://mtgtop8.com/event?e=79600&f=ST,Super Sunday RCQ #2,SCG CON Portland
5,2026-01-25,https://mtgtop8.com/event?e=79599&f=ST,Super Sunday RCQ #1,SCG CON Portland
6,2026-01-25,https://mtgtop8.com/event?e=79598&f=ST,Izzet Explosive Experiment Event,NaN
7,2026-01-25,https://mtgtop8.com/event?e=79462&f=ST,Regional Championship,SCG CON Portland
8,2026-01-24,https://mtgtop8.com/event?e=79569&f=ST,RCQ,SCG CON Portland
9,2026-01-24,https://mtgtop8.com/event?e=79428&f=ST,RCQ,"BB-Spiele (Rosenheim, Germany)"


5. Datatypes?

The last thing left to check is the data types for each of these columns and ensure they're right. 

In [169]:
display(all_tournaments_df.dtypes)

Date          datetime64[us]
Link                     str
Name                     str
Convention               str
dtype: object

They are! We're left with the final tournament data frame below.

In [170]:
display(all_tournaments_df.head(), all_tournaments_df.dtypes)

,Date,Link,Name,Convention
0,2026-01-31,https://mtgtop8.com/event?e=79854&f=ST,2nd Chance PTQ,"Pro Tour Lorwyn Eclipsed (Richmond, VA)"
1,2026-01-30,https://mtgtop8.com/event?e=79785&f=ST,Champions Cup Special Qualifier,Kawasaki (Japan)
2,2026-01-30,https://mtgtop8.com/event?e=79746&f=ST,Pro Tour Lorwyn Eclipsed,Richmond
3,2026-01-25,https://mtgtop8.com/event?e=79664&f=ST,ReCQ #2,SCG CON Portland
4,2026-01-25,https://mtgtop8.com/event?e=79600&f=ST,Super Sunday RCQ #2,SCG CON Portland


Date          datetime64[us]
Link                     str
Name                     str
Convention               str
dtype: object

## Scraping Deck Data

### Extracting

### Transforming

### Loading

# Exploratory Data Analysis

## Deck Archetypes

- explain how decks are named by archetypes and how even though individual cards may differ the deck still maintains a "gimmick" (*strategy* strengths and weaknesses)
- reason: examining these deck archetypes closer would indicate future features to explore (mana curve, card cost, creature vs spells)

### What was the most popular deck archetype?

### Which deck archetype was most common for each rank? 

### Does popular deck archetype align with the best for winning? 

## Deck Prices

### What was the average cost of a deck for each rank? 

### Which rank had the most spread in deck price?

## Rank Distribution

# References
- https://en.wikipedia.org/wiki/Magic:_The_Gathering
- https://pandas.pydata.org/docs/reference/api/pandas.Series.str.extract.html

In [38]:
!jupyter nbconvert --to html /Users/aortego1/Documents/YR1/S2026/CMPS-6790-DS/FinalNotebook/cmps-6790-final-tutorial/Milestone1_FD.ipynb

[NbConvertApp] Converting notebook /Users/aortego1/Documents/YR1/S2026/CMPS-6790-DS/FinalNotebook/cmps-6790-final-tutorial/Milestone1_FD.ipynb to html
[NbConvertApp] Writing 316375 bytes to /Users/aortego1/Documents/YR1/S2026/CMPS-6790-DS/FinalNotebook/cmps-6790-final-tutorial/Milestone1_FD.html
